In [21]:
from openai import OpenAI
from langsmith import Client
import os


In [25]:
client=Client(api_key="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")

In [26]:
datasets=client.read_dataset(
    dataset_name="rag_eval2"
)

In [27]:
datasets

Dataset(name='rag_eval2', description='Dataset for Evaluation Rag pipeline', data_type=<DataType.kv: 'kv'>, id=UUID('7f2ed3e7-ff32-4eb7-810c-0fde541fc996'), created_at=datetime.datetime(2026, 7, 8, 14, 50, 15, 304116, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 7, 8, 14, 50, 15, 304116, tzinfo=TzInfo(0)), example_count=27, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Windows-11-10.0.26200-SP0', 'sdk_version': '0.9.7', 'runtime_version': '3.14.5', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [28]:
examples_list = list(client.list_examples(dataset_id=datasets.id, limit=10))[0].inputs
examples_list

{'question': 'What are the specifications of the Gigabyte GeForce RTX 4070 Ti graphics card?'}

In [29]:
refrence_input=list(client.list_examples(dataset_id=datasets.id, limit=10))[0].inputs
refrence_output=list(client.list_examples(dataset_id=datasets.id, limit=10))[0].outputs

In [64]:
refrence_output

{'ground_truth': 'The Gigabyte RTX 4070 Ti features 12GB GDDR6X memory, 192-bit interface, 3X WINDFORCE fans, supports DLSS 3, ray tracing, and 4K gaming with a max resolution of 7680x4320.',
 'refrence_context_ids': ['B0BRR2R8HH'],
 'reference_description': ['Gigabyte GeForce RTX 4070 Ti Gaming OC 12G Graphics Card, 3X WINDFORCE Fans, 12GB 192-bit GDDR6X, GV-N407TGAMING OC-12GD Video CardPowered by NVIDIA DLSS 3.Digital max resolution: 7680x4320NVIDIA Ada Lovelace Streaming Multiprocessors: Up to 2x performance and power efficiency4th Generation Tensor Cores: Up to 2x AI performance3rd Generation RT Cores: Up to 2x ray tracing performancePowered by GeForce RTX 4070 Ti 12GBIntegrated with 12GB GDDR6X 192-bit memory interfaceWINDFORCE Cooling System, RGB Fusion, Dual BIOS, Protection metal back plate, Anti-Sag Bracket']}

## Rag Pipe Line

In [30]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct
load_dotenv()
client=OpenAI()





# Qdrant host: "http://qdrant:6333" inside docker-compose, "http://localhost:6333" locally.
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")

def get_embedding(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    
    return response.data[0].embedding

def reteriver_data(query, qdrant_client, k):
    query_embedding = get_embedding(query)

    result = qdrant_client.query_points(
        collection_name="Amazon_items_collection-00",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []
    retrieved_image_urls = []
    retrieved_prices = []

    for point in result.points:
        payload = point.payload or {}
        retrieved_context_ids.append(payload.get("parent_asin"))
        retrieved_context.append(payload.get("description", ""))
        retrieved_context_rating.append(payload.get("average_rating"))
        similarity_scores.append(point.score)
        retrieved_image_urls.append(payload.get("image_url", ""))
        retrieved_prices.append(payload.get("price"))

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores,
        retrieved_image_urls,
        retrieved_prices
    )


def process_context(context, ids, ratings, scores):
    formatted_context = ""

    for id, chunk, rating, score in zip(ids, context, ratings, scores):
        formatted_context += (
            f"- ID: {id}\n"
            f"  Description: {chunk}\n"
            f"  Rating: {rating}\n"
            f"  Similarity Score: {score:.4f}\n\n"
        )

    return formatted_context

def build_prompt(processed_context, question):
    prompt = f"""
You are a helpful AI shopping assistant.

Use ONLY the information provided in the retrieved product context below to answer the user's question.

Rules:
- Do not make up information.
- If the answer is not present in the context, reply:
  "I couldn't find that information in the retrieved products."
- Keep your answer concise and helpful.
- Mention product IDs when relevant.

======================
Retrieved Context:
{processed_context}
======================

User Question:
{question}

Answer:
"""

    return prompt

def generate_chat(prompt):
    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": prompt
            },
            
        ],
        temperature=0.2,
        max_tokens=300
    )

    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(QDRANT_URL)

    retrieved_context = reteriver_data(
        question,
        qdrant_client,
        top_k
    )
   
    processed_context = process_context(
        retrieved_context[0],
        retrieved_context[1],
        retrieved_context[2],
        retrieved_context[3]
    )

    prompt = build_prompt(
        processed_context,
        question
    )
    
    answer = generate_chat(prompt)
    final_result={
        "answer":answer,
        "Question":question,
        "reterived_context_ids":retrieved_context[1],
        "reterived_context":retrieved_context[0],
        "similaritry_ecore":retrieved_context[3]
    }
    return final_result


    



In [32]:
res=rag_pipeline("can i get some Charger",top_k=5)

In [33]:
res

{'answer': 'Yes, you can find chargers in the product listings. For example, ID B0CFG7V2QX offers a 20W USB C iPhone Charger Fast Charging Block with a 10FT long cable, and ID B0BV6PWVCG provides a 6FT USB C to Lightning Cable with fast charging capabilities.',
 'Question': 'can i get some Charger',
 'reterived_context_ids': ['B0BPLX388R',
  'B0CFG7V2QX',
  'B09X73T9DQ',
  'B0BV6PWVCG',
  'B0CCNXLP57'],
 'reterived_context': ['5pack iPhone Charger Lightning Cables for Phone 13 13pro Max 12 Mini 11 Se 10 X Xr Xs 8 7 6 6s, Apple Certified Original Braided Cord Multi-Color Lightening Wire Fast Charging Cargador Chord-3 6 10 FtAPPLE MFI CERTIFIED - Made with original Apple MFI Chip, full support with iOS versions. High quality, pull-resistance, fast charging, and transmission.USB A TO LIGHTNING - USB-A to USB-Lightning port cable matches with USB A Wall charger. you can charge your iPhone, iPad, iPod. But not fit for android micro or type c port phones. 【Please make sure this is not a USB-

In [37]:
from ragas.dataset_schema import SingleTurnSample 
from ragas.metrics import (
    ContextPrecision,  # Formerly IDBasedContextPrecision
    ContextRecall,     # Formerly IDBasedContextRecall
    Faithfulness,      # Fixed typo: 'FaithFullness' -> 'Faithfulness'
    ResponseRelevancy  # Fixed typo: 'ResponceRelevancy' -> 'ResponseRelevancy'
)


C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\3912272382.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\3912272382.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\3912272382.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\3912

In [38]:
import os

os.environ.pop("LANGCHAIN_TRACING_V2", None)
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ["LANGCHAIN_TRACING_V2"] = "false"
from ragas.llms  import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

In [44]:
ragas_llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
ragas_embedding=LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\340296611.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm=LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\340296611.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embedding=LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [40]:
res=rag_pipeline(refrence_input["question"])

In [41]:
res

{'answer': 'The Gigabyte GeForce RTX 4070 Ti Gaming OC 12G Graphics Card (ID: B0BRR2R8HH) features 12GB of GDDR6X memory, a 192-bit memory interface, and is powered by NVIDIA DLSS 3. It has a cooling system with 3 WINDFORCE fans, RGB Fusion, Dual BIOS, a protection metal back plate, and an anti-sag bracket. It supports digital max resolution up to 7680x4320 and offers up to 2x performance and power efficiency with NVIDIA Ada Lovelace Streaming Multiprocessors, 4th Generation Tensor Cores, and 3rd Generation RT Cores.',
 'Question': 'What are the specifications of the Gigabyte GeForce RTX 4070 Ti graphics card?',
 'reterived_context_ids': ['B0BRR2R8HH',
  'B08SVGR1W8',
  'B0CHGXHB81',
  'B0BQZ23TJL',
  'B0BVX6BBMJ'],
 'reterived_context': ['Gigabyte GeForce RTX 4070 Ti Gaming OC 12G Graphics Card, 3X WINDFORCE Fans, 12GB 192-bit GDDR6X, GV-N407TGAMING OC-12GD Video CardPowered by NVIDIA DLSS 3.Digital max resolution: 7680x4320NVIDIA Ada Lovelace Streaming Multiprocessors: Up to 2x perfo

In [47]:
async def ragas_faithfullness(run,example):
    sample=SingleTurnSample(
        user_input=run['Question'],
        response=run['answer'],
        retrieved_contexts=run['reterived_context']
    )
    scorer=Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)


In [48]:
result=await ragas_faithfullness(res,"")

In [49]:
result

1.0

In [52]:
async def ragas_responcereleveancy(run,example):
    sample=SingleTurnSample(
        user_input=run['Question'],
        response=run['answer'],
        retrieved_contexts=run['reterived_context']
    )
    scorer=ResponseRelevancy(llm=ragas_llm,embeddings=ragas_embedding)
    return await scorer.single_turn_ascore(sample)

In [53]:
await ragas_responcereleveancy(res,"")

np.float64(0.8377174055347614)

In [67]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision

C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\1969490736.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision


In [75]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall

async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["reterived_context_ids"],
        reference_context_ids=example["refrence_context_ids"],
    )

    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\3985038912.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall
C:\Users\jaysi\AppData\Local\Temp\ipykernel_32188\3985038912.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision,IDBasedContextRecall


In [76]:
await ragas_context_precision_id_based(res,refrence_output)

0.2

In [77]:

async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["reterived_context_ids"],
        reference_context_ids=example["refrence_context_ids"],
    )

    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

In [79]:
await ragas_context_recall_id_based(res, refrence_output)

1.0